# QuantAlpha AI — Step 5: Explainable AI + Entry/Target/Stop-Loss

**What we've validated so far, honestly:**
- Pure technical timing: no real edge
- Sector momentum: no real edge (properly time-split tested)
- Fundamental quality (F-Score + ROCE): promising edge (+5.28 pts on 12-month returns), but with
  a look-ahead bias caveat since we only have current, not historical, fundamentals

**This notebook builds on the fundamental quality signal** — the one real lead we have — and
wraps it in:
1. A human-readable explanation (the "Recommendation: BUY, Confidence: X%, Reasons: ..." format
   from your original doc)
2. Entry zone, target price, and stop-loss levels using ATR-based risk calculation
3. **A confidence label that's honest about validation status** — this is what makes the
   platform trustworthy rather than overselling an unproven signal


## 1. Connect to Drive + load everything built so far

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')

import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

fundamental_scores = pd.read_csv('data/fundamental_scores.csv')
fundamental_scores['symbol_short'] = fundamental_scores['symbol'].str.replace('.NS', '', regex=False)

event_classification = pd.read_csv('data/event_classification.csv') if os.path.exists('data/event_classification.csv') else pd.DataFrame()
sector_df = pd.read_csv('data/sector_mapping.csv') if os.path.exists('data/sector_mapping.csv') else pd.DataFrame()

print(f"Fundamental scores: {len(fundamental_scores)} stocks")
print(f"Event classifications: {len(event_classification)} events")
print(f"Sector mappings: {len(sector_df)} stocks")


Mounted at /content/drive
Fundamental scores: 200 stocks
Event classifications: 92 events
Sector mappings: 200 stocks


## 2. Build the master stock profile (latest technical + fundamental + sector + events)

In [2]:
technical_files = glob.glob('data/technical/*.parquet')
symbols = sorted(f.split('/')[-1].replace('.parquet', '') for f in technical_files)

def get_latest_technical(symbol):
    df = pd.read_parquet(f"data/technical/{symbol}.parquet")
    if df.empty:
        return None
    latest = df.iloc[-1].to_dict()
    latest['symbol_short'] = symbol
    latest['history_df'] = None  # placeholder, not stored per-row
    return latest

technical_latest = [row for row in (get_latest_technical(s) for s in symbols) if row]
technical_df = pd.DataFrame(technical_latest)

master = technical_df.merge(fundamental_scores, on='symbol_short', how='left')
if len(sector_df):
    master = master.merge(sector_df[['symbol_short', 'sector']], on='symbol_short', how='left')
if len(event_classification):
    event_classification['symbol_short'] = event_classification['symbol'].str.replace('.NS', '', regex=False)
    event_summary = event_classification.groupby('symbol_short').agg(
        latest_verdict=('verdict', 'first'),
        has_structural_flag=('verdict', lambda x: any('STRUCTURAL' in v for v in x))
    ).reset_index()
    master = master.merge(event_summary, on='symbol_short', how='left')
    master['has_structural_flag'] = master['has_structural_flag'].fillna(False)
else:
    master['has_structural_flag'] = False
    master['latest_verdict'] = None

print(f"Master profile built: {len(master)} stocks")


Master profile built: 200 stocks


## 3. Percentile scoring (reused from Step 3)

In [3]:
def percentile_score(series):
    ranked = series.rank(pct=True, na_option='keep') * 100
    return ranked.fillna(50)

master['fundamental_score'] = (
    0.35 * percentile_score(master['piotroski_f_score']) +
    0.25 * percentile_score(master['roce']) +
    0.20 * percentile_score(master['revenue_growth']) +
    0.20 * percentile_score(master['net_income_growth'])
)
master['cashflow_score'] = (
    0.5 * percentile_score(master['ocf_margin']) +
    0.5 * percentile_score(master['fcf_margin'])
)

# Overall Long-Term score leans heavily on the ONE validated signal: fundamental quality
master['longterm_overall_score'] = (
    0.55 * master['fundamental_score'] +
    0.30 * master['cashflow_score'] +
    0.15 * percentile_score(master['ADX_14'])  # small technical component, low weight given no proven edge
)

print("Scores computed.")


Scores computed.


## 4. Confidence labeling — the honest part
Confidence isn't just "how high is the score" — it also reflects HOW VALIDATED the underlying
signal is. A stock can have a great fundamental score but still only get "Moderate" confidence
if there's a structural risk flag or if data was incomplete.


In [4]:
def compute_confidence(row):
    base_confidence = row['longterm_overall_score']  # 0-100 scale already

    # Penalize if structural news risk flagged
    if row['has_structural_flag']:
        base_confidence *= 0.6

    # Penalize if fundamental data was incomplete (fewer than 7 of 9 Piotroski checks available)
    if pd.notna(row.get('piotroski_checks_available')) and row['piotroski_checks_available'] < 7:
        base_confidence *= 0.85

    return round(min(base_confidence, 95), 1)  # cap at 95 -- never claim near-certainty

def confidence_label(score):
    if score >= 75:
        return "High"
    elif score >= 55:
        return "Moderate"
    else:
        return "Low"

master['confidence_score'] = master.apply(compute_confidence, axis=1)
master['confidence_label'] = master['confidence_score'].apply(confidence_label)
print("Confidence scores computed.")
master[['symbol_short', 'longterm_overall_score', 'confidence_score', 'confidence_label']].describe()


Confidence scores computed.


,longterm_overall_score,confidence_score
count,200.000000,200.000000
mean,50.250000,46.685500
std,16.547082,17.729946
min,11.003676,6.600000
25%,38.574165,33.050000
50%,51.080376,47.650000
75%,60.890286,58.675000
max,90.913413,88.800000


## 5. Entry Zone / Target / Stop-Loss (ATR-based)
Uses Average True Range (ATR) — a standard volatility measure — to set risk levels that adapt to
each stock's own volatility, rather than a fixed percentage for every stock.


In [5]:
def calculate_levels(row):
    price = row['Close']
    atr = row['ATR_14']
    if pd.isna(price) or pd.isna(atr) or atr <= 0:
        return pd.Series({'entry_low': np.nan, 'entry_high': np.nan, 'stop_loss': np.nan,
                          'target_1': np.nan, 'target_2': np.nan})

    entry_low = price - (0.5 * atr)
    entry_high = price + (0.5 * atr)
    stop_loss = price - (2.0 * atr)     # 2x ATR below entry -- standard risk buffer
    target_1 = price + (3.0 * atr)      # ~1.5:1 reward-to-risk
    target_2 = price + (5.0 * atr)      # stretch target for strong setups

    return pd.Series({
        'entry_low': round(entry_low, 2),
        'entry_high': round(entry_high, 2),
        'stop_loss': round(stop_loss, 2),
        'target_1': round(target_1, 2),
        'target_2': round(target_2, 2)
    })

levels = master.apply(calculate_levels, axis=1)
master = pd.concat([master, levels], axis=1)
print("Entry/target/stop-loss levels calculated.")


Entry/target/stop-loss levels calculated.


## 6. Explainable AI narrative generator
Turns the scores into a readable recommendation, matching the format from your original doc —
but with an honest confidence framing baked in.


In [6]:
def generate_explanation(row):
    reasons = []

    if row['piotroski_f_score'] >= 7:
        reasons.append(f"Strong Piotroski F-Score ({int(row['piotroski_f_score'])}/9) — healthy profitability and efficiency trend")
    elif row['piotroski_f_score'] <= 3:
        reasons.append(f"Weak Piotroski F-Score ({int(row['piotroski_f_score'])}/9) — caution on financial health")

    if pd.notna(row['roce']) and row['roce'] > 0.15:
        reasons.append(f"Strong ROCE ({row['roce']*100:.1f}%) — efficient use of capital")

    if pd.notna(row['revenue_growth']) and row['revenue_growth'] > 0.10:
        reasons.append(f"Revenue growing at {row['revenue_growth']*100:.1f}% year-over-year")

    if pd.notna(row['fcf_margin']) and row['fcf_margin'] > 0.10:
        reasons.append(f"Healthy free cash flow margin ({row['fcf_margin']*100:.1f}%)")

    if row['has_structural_flag']:
        reasons.append("⚠️ Recent news flagged a possible structural concern — verify before acting")

    if pd.notna(row.get('sector')):
        reasons.append(f"Sector: {row['sector']}")

    verdict = "BUY (Long-Term candidate)" if row['longterm_overall_score'] >= 65 and not row['has_structural_flag'] else \
              "WATCH" if row['longterm_overall_score'] >= 50 else "AVOID"

    disclaimer = ("Note: this recommendation is based on a fundamental-quality signal that showed "
                  "a real backtested edge, but using current fundamentals against historical price "
                  "data (a limitation, not a guarantee). Treat as one input, not financial advice.")

    return {
        'symbol': row['symbol_short'],
        'recommendation': verdict,
        'confidence': f"{row['confidence_score']}% ({row['confidence_label']})",
        'reasons': reasons,
        'entry_zone': f"₹{row['entry_low']} - ₹{row['entry_high']}" if pd.notna(row['entry_low']) else "N/A",
        'stop_loss': f"₹{row['stop_loss']}" if pd.notna(row['stop_loss']) else "N/A",
        'target_1': f"₹{row['target_1']}" if pd.notna(row['target_1']) else "N/A",
        'target_2': f"₹{row['target_2']}" if pd.notna(row['target_2']) else "N/A",
        'disclaimer': disclaimer
    }

# Example: show explanation for top 5 Long-Term candidates
top5 = master.nlargest(5, 'longterm_overall_score')
for _, row in top5.iterrows():
    exp = generate_explanation(row)
    print(f"\n{'='*70}")
    print(f"SYMBOL: {exp['symbol']}")
    print(f"Recommendation: {exp['recommendation']}")
    print(f"Confidence: {exp['confidence']}")
    print(f"Current Price: Rs.{row['Close']:.2f}")
    print(f"Entry Zone: {exp['entry_zone']}")
    print(f"Stop-Loss: {exp['stop_loss']}")
    print(f"Target 1: {exp['target_1']}  |  Target 2: {exp['target_2']}")
    print(f"\nReasons:")
    for r in exp['reasons']:
        print(f"  - {r}")
    print(f"\n{exp['disclaimer']}")



SYMBOL: NATIONALUM
Recommendation: WATCH
Confidence: 54.5% (Low)
Current Price: Rs.347.60
Entry Zone: ₹340.89 - ₹354.31
Stop-Loss: ₹320.78
Target 1: ₹387.84  |  Target 2: ₹414.66

Reasons:
  - Strong Piotroski F-Score (9/9) — healthy profitability and efficiency trend
  - Strong ROCE (37.0%) — efficient use of capital
  - Revenue growing at 27.5% year-over-year
  - Healthy free cash flow margin (27.6%)
  - ⚠️ Recent news flagged a possible structural concern — verify before acting
  - Sector: Basic Materials

Note: this recommendation is based on a fundamental-quality signal that showed a real backtested edge, but using current fundamentals against historical price data (a limitation, not a guarantee). Treat as one input, not financial advice.

SYMBOL: MCX
Recommendation: BUY (Long-Term candidate)
Confidence: 88.8% (High)
Current Price: Rs.2814.00
Entry Zone: ₹2763.46 - ₹2864.54
Stop-Loss: ₹2611.83
Target 1: ₹3117.26  |  Target 2: ₹3319.43

Reasons:
  - Strong Piotroski F-Score (9/9) 

## 7. Save full ranked output with explanations for Top 25

In [7]:
top25 = master.nlargest(25, 'longterm_overall_score')
explanations = [generate_explanation(row) for _, row in top25.iterrows()]

output_rows = []
for exp in explanations:
    output_rows.append({
        'symbol': exp['symbol'],
        'recommendation': exp['recommendation'],
        'confidence': exp['confidence'],
        'entry_zone': exp['entry_zone'],
        'stop_loss': exp['stop_loss'],
        'target_1': exp['target_1'],
        'target_2': exp['target_2'],
        'reasons': ' | '.join(exp['reasons'])
    })

output_df = pd.DataFrame(output_rows)
output_df.to_csv('data/longterm_recommendations_explained.csv', index=False)
print("Saved: data/longterm_recommendations_explained.csv")
output_df.head(10)


Saved: data/longterm_recommendations_explained.csv


,symbol,recommendation,confidence,entry_zone,stop_loss,target_1,target_2,reasons
0,NATIONALUM,WATCH,54.5% (Low),₹340.89 - ₹354.31,₹320.78,₹387.84,₹414.66,Strong Piotroski F-Score (9/9) — healthy profi...
1,MCX,BUY (Long-Term candidate),88.8% (High),₹2763.46 - ₹2864.54,₹2611.83,₹3117.26,₹3319.43,Strong Piotroski F-Score (9/9) — healthy profi...
2,GMRAIRPORT,BUY (Long-Term candidate),88.2% (High),₹111.37 - ₹114.23,₹107.08,₹121.38,₹127.1,Strong Piotroski F-Score (9/9) — healthy profi...
3,ENRIN,BUY (Long-Term candidate),82.6% (High),₹3207.3 - ₹3355.7,₹2984.71,₹3726.69,₹4023.48,Strong Piotroski F-Score (9/9) — healthy profi...
4,HINDZINC,BUY (Long-Term candidate),82.2% (High),₹528.78 - ₹544.32,₹505.47,₹583.17,₹614.26,Strong Piotroski F-Score (8/9) — healthy profi...
5,BSE,BUY (Long-Term candidate),82.1% (High),₹3758.65 - ₹3874.55,₹3584.78,₹4164.32,₹4396.14,Strong Piotroski F-Score (9/9) — healthy profi...
6,LUPIN,BUY (Long-Term candidate),81.5% (High),₹2452.49 - ₹2499.51,₹2381.95,₹2617.08,₹2711.14,Strong Piotroski F-Score (8/9) — healthy profi...
7,LAURUSLABS,BUY (Long-Term candidate),80.0% (High),₹1522.52 - ₹1562.08,₹1463.2,₹1660.95,₹1740.06,Strong Piotroski F-Score (9/9) — healthy profi...
8,GVT&D,BUY (Long-Term candidate),79.1% (High),₹4276.09 - ₹4526.91,₹3899.87,₹5153.94,₹5655.57,Strong Piotroski F-Score (8/9) — healthy profi...
9,OFSS,BUY (Long-Term candidate),77.8% (High),₹11076.88 - ₹11419.12,₹10563.52,₹12274.73,₹12959.21,Strong Piotroski F-Score (8/9) — healthy profi...


## 8. Summary + honest project status

**What this platform can currently do, honestly:**
- Score all 200 Nifty 200 stocks on fundamental quality (F-Score, ROCE, cash flow, growth)
- Generate a human-readable BUY/WATCH/AVOID recommendation with specific reasons
- Calculate volatility-adjusted entry/target/stop-loss levels
- Flag stocks with recent negative news and classify likely temporary vs structural
- Label its own confidence honestly, including when data was incomplete

**What it does NOT yet do (be upfront about this in any interview/demo):**
- Swing Trading mode — tested, showed no reliable technical edge; treat as experimental only
- Sector-adjusted scoring — tested, showed no reliable edge; not built into scoring
- Fully rigorous backtesting of the combined score — needs historical point-in-time fundamentals
  (a bigger future data engineering project, likely needing paid/scraped historical data)
- Real-time data — everything here is a snapshot from early July 2026, not live-updating

**This honesty is itself a strength.** A platform that clearly states what's validated vs.
experimental is more credible — in an interview and in reality — than one that claims uniform
90%+ accuracy across every feature.

**Natural next steps (future sessions, not urgent before July 15):**
- Extend price history to 7-10 years to test signals across bull AND bear markets
- Source historical point-in-time fundamentals for a rigorous quality-signal backtest
- Build the FastAPI backend to serve these CSVs as an API
- Push everything to GitHub with a clear README documenting validated vs. experimental components
